# StyleMatch all-source corpus, GPU training, and fast index

This notebook is the Colab-only build stage. It fetches every source currently available from the registry, creates one combined parquet file for both literary and rhetorical corpora (no per-chunk text files), optionally fine-tunes mStyleDistance, and builds a cached style + topic profile index. Translation is not used in the default path.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
assert (REPO / 'scripts/multilingual_style_index.py').exists(), REPO
%cd $REPO
print('repo:', Path.cwd())

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install requests beautifulsoup4 pandas pyarrow sentence-transformers scikit-learn

## 1. Fetch every available source

The first command searches Gutenberg for every registry author in both corpora, including registry-only candidates. The second fetches the curated Chinese, Japanese, French, German, and Russian originals. A missing result is reported and skipped; it is never replaced with a translation.

In [ ]:
OLD_ROOT = Path('/content/drive/MyDrive/stylematch_v1')
if OLD_ROOT.exists():
    subprocess.run([sys.executable, 'scripts/merge_existing_corpus.py', '--source-root', str(OLD_ROOT), '--corpus', 'both'], check=True)
else:
    print('No legacy stylematch_v1 directory found; continuing with repo sources.')

!python scripts/fetch_gutendex.py --corpus both --language en --max-works 0
!python scripts/fetch_multilingual_sources.py --language zh --language ja --language fr --language de --language ru --skip-existing

In [ ]:
MANIFEST = REPO / 'data/source_registry/source_manifest.csv'
!python scripts/import_source_manifest.py "{MANIFEST}" --append
chunks_path = REPO / 'data/all/meta/all_sources_chunks.parquet'
!python scripts/build_chunk_parquet_from_sources.py --corpus both --output "{chunks_path}" --coverage-output data/all/meta/all_sources_coverage.json --min-sources 3 --min-chunks 30
heldout_path = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
!python scripts/make_source_heldout_splits.py --input "{chunks_path}" --output "{heldout_path}" --report data/all/meta/all_source_heldout_report.json

In [ ]:
import json, pandas as pd
chunks_path = REPO / 'data/all/meta/all_sources_chunks.parquet'
chunks = pd.read_parquet(chunks_path)
print(chunks.shape)
display(chunks.groupby(['language', 'author_or_speaker']).size().sort_values())
coverage = json.loads((REPO / 'data/all/meta/all_sources_coverage.json').read_text())
print('authors:', coverage['n_authors'], 'author-language profiles:', coverage['n_author_language_profiles'], 'sources:', coverage['n_sources'])
display(coverage['not_ready'])
heldout_report = json.loads((REPO / 'data/all/meta/all_source_heldout_report.json').read_text())
print('source-heldout eligible profiles:', heldout_report['eligible_authors'], 'profiles:', heldout_report['n_author_language_profiles'], 'sources:', heldout_report['n_sources'])

## 2. Optional GPU fine-tuning

Run this after the source coverage cell. Every author with at least one source enters the combined profile index. The adapter objective uses same-author, different-source pairs and in-batch negatives; authors with only one source cannot form a valid source-separated positive pair, so they remain in retrieval but are excluded from this specific fine-tuning loss. It never translates or generates source text.

In [ ]:
# Train on every author-language profile with at least two sources; the pair sampler enforces source separation.
!python scripts/finetune_multilingual_style.py --input "{chunks_path}" --output-dir artifacts/mstyledistance_stylematch_v1 --pairs-per-author 500 --batch-size 32 --epochs 1 --device cuda

## 3. Build the cached fast index

The first build encodes corpus chunks on GPU. Re-running it reuses `chunk_embeddings.npz` and `topic_chunk_embeddings.npz`; only new chunk IDs are encoded. The topic model is physically separate from the style model.

In [ ]:
!python scripts/style_embedding_recall.py --input "{heldout_path}" --out-dir artifacts/source_heldout_eval_v1 --model-name artifacts/mstyledistance_stylematch_v1 --batch-size 128 --train-cap 300 --eval-splits dev,test --device cuda
!python scripts/multilingual_style_index.py build --input "{chunks_path}" --out-dir artifacts/multilingual_style_index_v1 --model-name artifacts/mstyledistance_stylematch_v1 --topic-model-name intfloat/multilingual-e5-base --batch-size 128 --per-source-cap 50 --profile-cap 600 --device cuda

In [ ]:
!python scripts/multilingual_style_index.py benchmark --index-dir artifacts/multilingual_style_index_v1 --language en --mode within --text "The institution changed slowly, while ordinary people learned to live with its contradictions." --runs 20 --device cuda
!python scripts/multilingual_style_index.py query --index-dir artifacts/multilingual_style_index_v1 --language en --mode within --text "The institution changed slowly, while ordinary people learned to live with its contradictions." --top-k 3 --device cuda

## 4. Artifact check

Keep the parquet, model directory, and index directory in Drive. Do not save chunk-level `.txt` files. The index directory is what the future web app loads.

In [ ]:
!du -sh data/all/meta/all_sources_chunks.parquet artifacts/mstyledistance_stylematch_v1 artifacts/multilingual_style_index_v1
print('training/index artifacts are under:', REPO / 'artifacts')